# Mecanismo de Busca em PLN — Netflix Shows (Busca Ingênua vs. BM25 vs. WAND)

Notebook auto-contido para execução no **Google Colab**.
- **Dataset:** [Netflix Movies and TV Shows (Kaggle)](https://www.kaggle.com/datasets/shivamb/netflix-shows)
- **Mecanismos:** Busca Ingênua (varredura linear), Okapi BM25 Puro e WAND (*Weak AND*).

In [ ]:
# 1. Download e extração do dataset do Kaggle
!mkdir -p data
!curl -L -o data/netflix-shows.zip https://www.kaggle.com/api/v1/datasets/download/shivamb/netflix-shows
!unzip -o data/netflix-shows.zip -d data/
!ls -lh data/

In [ ]:
# 2. Pré-processamento, Busca Ingênua, BM25 Puro e WAND
from __future__ import annotations
import heapq, math, re, statistics, time, unicodedata
from bisect import bisect_left
from collections import Counter
from dataclasses import dataclass
from operator import attrgetter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

_TOKEN_RE = re.compile(r"[a-z0-9]+")
STOPWORDS = frozenset({
    "a", "about", "above", "after", "again", "against", "all", "am", "an",
    "and", "any", "are", "as", "at", "be", "because", "been", "before",
    "being", "below", "between", "both", "but", "by", "can", "could",
    "did", "do", "does", "doing", "down", "during", "each", "few", "for",
    "from", "further", "had", "has", "have", "having", "he", "her", "here",
    "hers", "herself", "him", "himself", "his", "how", "i", "if", "in",
    "into", "is", "it", "its", "itself", "just", "me", "more", "most",
    "my", "myself", "no", "nor", "not", "now", "of", "off", "on", "once",
    "only", "or", "other", "our", "ours", "ourselves", "out", "over",
    "own", "same", "she", "should", "so", "some", "such", "than", "that",
    "the", "their", "theirs", "them", "themselves", "then", "there",
    "these", "they", "this", "those", "through", "to", "too", "under",
    "until", "up", "very", "was", "we", "were", "what", "when", "where",
    "which", "while", "who", "whom", "why", "will", "with", "would", "you",
    "your", "yours", "yourself", "yourselves",
})

def tokenize(text: str) -> list[str]:
    decomposed = unicodedata.normalize("NFKD", text.lower())
    clean = "".join(ch for ch in decomposed if not unicodedata.combining(ch))
    return [t for t in _TOKEN_RE.findall(clean) if t not in STOPWORDS]

@dataclass(slots=True)
class NetflixDocument:
    doc_id: int
    title: str
    media_type: str
    release_year: int
    listed_in: str
    tokens: list[str]

df = pd.read_csv("data/netflix_titles.csv").fillna("")
docs = []
for idx, row in enumerate(df.itertuples(index=False)):
    raw = f"{row.title} {row.listed_in} {row.director} {row.cast} {row.country} {row.description}"
    docs.append(NetflixDocument(idx, str(row.title), str(row.type), int(row.release_year), str(row.listed_in), tokenize(raw)))

print(f"Documentos carregados: {len(docs)}")

In [ ]:
# 3. Implementação dos Mecanismos
class NaiveSearchEngine:
    def __init__(self, documents: list[NetflixDocument]):
        self.documents = documents

    def search(self, query: str, k: int = 10):
        t0 = time.perf_counter_ns()
        q_counts = Counter(tokenize(query))
        q_set = set(q_counts)
        scored = []
        for doc in self.documents:
            s = sum(q_counts[tok] for tok in doc.tokens if tok in q_set)
            if s > 0:
                scored.append((float(s), -doc.doc_id))
        top = heapq.nlargest(k, scored)
        ms = (time.perf_counter_ns() - t0) / 1e6
        return [(-d, sc) for sc, d in top], ms, len(self.documents)

@dataclass(slots=True)
class PostingList:
    doc_ids: list[int]
    scores: list[float]
    max_score: float

class BM25Index:
    def __init__(self, documents: list[NetflixDocument], k1: float = 1.5, b: float = 0.75):
        self.documents = documents
        self.num_docs = len(documents)
        self.doc_lengths = [len(d.tokens) for d in documents]
        self.avgdl = sum(self.doc_lengths) / self.num_docs
        self.postings: dict[str, PostingList] = {}
        raw = {}
        for d in documents:
            for term, tf in Counter(d.tokens).items():
                raw.setdefault(term, []).append((d.doc_id, tf))
        for term, entries in raw.items():
            df_t = len(entries)
            idf = math.log(1.0 + (self.num_docs - df_t + 0.5) / (df_t + 0.5))
            d_ids, scs, max_s = [], [], 0.0
            for did, tf in entries:
                norm = k1 * (1.0 - b + b * (self.doc_lengths[did] / self.avgdl))
                s = idf * (tf * (k1 + 1.0)) / (tf + norm)
                d_ids.append(did)
                scs.append(s)
                if s > max_s:
                    max_s = s
            self.postings[term] = PostingList(d_ids, scs, max_s)

class BM25SearchEngine:
    def __init__(self, index: BM25Index):
        self.index = index

    def search(self, query: str, k: int = 10):
        t0 = time.perf_counter_ns()
        doc_scores: dict[int, float] = {}
        for term, q_tf in sorted(Counter(tokenize(query)).items()):
            p = self.index.postings.get(term)
            if p is None:
                continue
            for did, s in zip(p.doc_ids, p.scores):
                doc_scores[did] = doc_scores.get(did, 0.0) + s * q_tf
        top = heapq.nlargest(k, ((sc, -did) for did, sc in doc_scores.items()))
        ms = (time.perf_counter_ns() - t0) / 1e6
        return [(-d, sc) for sc, d in top], ms, len(doc_scores)

@dataclass(slots=True)
class _Cursor:
    term_order: int
    doc_ids: list[int]
    scores: list[float]
    q_tf: int
    ub: float
    length: int
    pos: int = 0
    doc_id: int = 0
    def __post_init__(self): self.doc_id = self.doc_ids[0]
    def advance(self):
        self.pos += 1
        if self.pos >= self.length: return False
        self.doc_id = self.doc_ids[self.pos]
        return True
    def seek(self, target: int):
        np_ = bisect_left(self.doc_ids, target, self.pos)
        if np_ >= self.length:
            self.pos = self.length
            return False
        self.pos = np_
        self.doc_id = self.doc_ids[np_]
        return True

class WANDSearchEngine:
    def __init__(self, index: BM25Index):
        self.index = index

    def search(self, query: str, k: int = 10):
        t0 = time.perf_counter_ns()
        cursors = []
        for order, (term, q_tf) in enumerate(sorted(Counter(tokenize(query)).items())):
            p = self.index.postings.get(term)
            if p and p.doc_ids:
                cursors.append(_Cursor(order, p.doc_ids, p.scores, q_tf, p.max_score * q_tf, len(p.doc_ids)))
        heap, docs_eval = [], 0
        sort_key = attrgetter("doc_id")
        while cursors:
            cursors.sort(key=sort_key)
            if len(heap) < k:
                pivot_idx = 0
            else:
                th, ub_sum, pivot_idx = heap[0][0], 0.0, -1
                for idx, c in enumerate(cursors):
                    ub_sum += c.ub
                    if ub_sum + 1e-12 >= th:
                        pivot_idx = idx
                        break
                if pivot_idx == -1:
                    break
            pivot_doc = cursors[pivot_idx].doc_id
            if cursors[0].doc_id == pivot_doc:
                matched, next_c, i = [], [], 0
                while i < len(cursors) and cursors[i].doc_id == pivot_doc:
                    c = cursors[i]
                    matched.append((c.term_order, c.scores[c.pos] * c.q_tf))
                    if c.advance(): next_c.append(c)
                    i += 1
                next_c.extend(cursors[i:])
                cursors = next_c
                if len(matched) > 2: matched.sort(key=lambda x: x[0])
                sc = sum(v for _, v in matched)
                docs_eval += 1
                cand = (sc, -pivot_doc)
                if len(heap) < k: heapq.heappush(heap, cand)
                elif cand > heap[0]: heapq.heapreplace(heap, cand)
            else:
                pick = 0
                best_ub = cursors[0].ub
                for idx in range(1, pivot_idx):
                    if cursors[idx].doc_id < pivot_doc and cursors[idx].ub > best_ub:
                        best_ub = cursors[idx].ub
                        pick = idx
                if not cursors[pick].seek(pivot_doc):
                    cursors.pop(pick)
        heap.sort(reverse=True)
        ms = (time.perf_counter_ns() - t0) / 1e6
        return [(-d, sc) for sc, d in heap], ms, docs_eval

naive = NaiveSearchEngine(docs)
index = BM25Index(docs)
bm25 = BM25SearchEngine(index)
wand = WANDSearchEngine(index)
print(f"Índice pronto: {len(index.postings)} termos únicos.")

In [ ]:
# 4. Benchmark Rápido e Verificação de Equivalência BM25 == WAND
queries = [
    "serial killer", "world war", "space alien adventure", "true crime murder",
    "indian bollywood drama romance", "united states documentary music",
    "international drama family love story", "united states comedy drama romance family",
    "united states international drama comedy action crime thriller",
]
rows = []
for q in queries:
    r_n, ms_n, d_n = naive.search(q, k=10)
    r_b, ms_b, d_b = bm25.search(q, k=10)
    r_w, ms_w, d_w = wand.search(q, k=10)
    assert [x[0] for x in r_b] == [x[0] for x in r_w], f"Divergência em {q}"
    rows.append({
        "query": q, "naive_ms": round(ms_n, 2), "bm25_ms": round(ms_b, 3), "wand_ms": round(ms_w, 3),
        "bm25_docs": d_b, "wand_docs": d_w, "poda_%": round((1 - d_w / d_b) * 100, 1)
    })
res_df = pd.DataFrame(rows)
display(res_df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(["Ingênua", "BM25", "WAND"], [res_df["naive_ms"].mean(), res_df["bm25_ms"].mean(), res_df["wand_ms"].mean()], color=["#7F8C8D", "#E50914", "#27AE60"])
ax1.set_yscale("log"); ax1.set_title("Latência Média (ms, log)")
ax2.bar(["Ingênua", "BM25", "WAND"], [len(docs), res_df["bm25_docs"].mean(), res_df["wand_docs"].mean()], color=["#7F8C8D", "#E50914", "#27AE60"])
ax2.set_title("Documentos Avaliados / Consulta (média)")
plt.tight_layout(); plt.show()